<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/Estimate_the_GPU_memory_required_to_train_a_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Before you can perform a single calculation, all of the necessary data must be loaded into the GPU's memory.

This includes the

    - model's parameters,
    - the input data, and
    - temporary values like gradients, optimizer states, and activations.

In [1]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

from ai_foundations.utils import formatting # For formatting memory estimates.
# For checking your solutions.
from ai_foundations.feedback.course_7 import memory as feedback

# Used to format the results of calculations.
from IPython.display import display, HTML

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-z8opnwb8
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-z8opnwb8
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Model parameters (4 billion).
PARAM_COUNT = 4e9

# Precision (bytes per parameter).
# For 32-bit (FP32), each parameter requires 4 bytes of storage.
BYTES_PER_PARAMETER = 4

# Activation formula constants.
BATCH_SIZE = 8
MAX_LENGTH = 1024 # Maximum sequence length.
NUM_LAYERS = 32
EMBEDDING_DIM = 2560

The memory required is the total number of parameters multiplied by the number of bytes each parameter occupies.

**1. Calculate parameter memory**

In [3]:
def calculate_param_memory(param_count: float, bytes_per_param: int) -> float:
    """Calculates the memory in GB required to store the model parameters.

    Args:
      param_count: The total number of parameters in the model.
      bytes_per_param: The number of bytes used to store a single parameter.

    Returns:
      The total memory required for parameters, in gigabytes.
    """
    total_bytes = param_count * bytes_per_param

    return formatting.bytes_to_gb(total_bytes)

In [4]:
feedback.test_calculate_param_memory(calculate_param_memory)

✅ Nice! Your answer looks correct.


**2. Calculate input data memory**

- During training, fine-tuning, or inference, must also load the **entire batch** of input data into the GPU's memory.
    -  compute how much memory is required to store the input for one batch


- the function calculate_input_data_memory below.

This function should return the number of bytes required to store the input to a model.

Hints:

    - The GPU processes a whole batch of training examples at once.
    
    - To find the total memory for the entire input batch, need to determine the number of bytes in a batch.

        - A batch contains batch_size examples of length max_length tokens.
        
        - Recall that each sequence has max_length because we pad and truncate it to length max_length.

        - One input tokens requires bytes_per_token_id bytes of memory.


In [5]:
def calculate_input_data_memory(
    batch_size: int, max_length: int, bytes_per_token_id: int
) -> float:
    """Calculates the memory in GB required to store a batch of input token IDs.

    Args:
      batch_size: The number of sequences in a single batch.
      max_length: The length of each sequence in tokens after padding.
      bytes_per_token_id: The number of bytes for each token ID.

    Returns:
      The total memory required for the input data batch, in gigabytes.
    """
    total_bytes = batch_size * max_length * bytes_per_token_id

    return formatting.bytes_to_gb(total_bytes)

In [6]:
feedback.test_calculate_input_data_memory(calculate_input_data_memory)

✅ Nice! Your answer looks correct.


**3. Calculate gradient memory**

- During the backward pass,
    - the optimizer computes a gradient that indicates how to update the parameters.
    
    - This gradient has one value for each model parameter and since the gradient is computed in the GPU, this also needs to be stored in the GPU memory.


- For each parameter in the model, store one number as part of the gradient.
    - The total memory required is therefore the total number of parameters multiplied by the number of bytes each parameter occupies.

In [7]:
def calculate_gradient_memory(param_count: float, bytes_per_param: int) -> float:
    """Calculates the memory in GB required to store the gradients.

    Args:
      param_count: The total number of parameters in the model.
      bytes_per_param: The number of bytes used to store a single parameter.

    Returns:
      The total memory required for gradients, in gigabytes.
    """
    total_bytes = param_count * bytes_per_param

    return formatting.bytes_to_gb(total_bytes)

In [8]:
feedback.test_calculate_gradient_memory(calculate_gradient_memory)

✅ Nice! Your answer looks correct.


**4. Calculate optimizer memory**

- Modern optimizers such as Adam also keep information that is used to compute learning rates for each individual parameter.

- This information also needs to be stored on the GPU.


- For each parameter in the model, an optimizer such as Adam stores information that is used to compute the learning rate for that specific parameter.

- The memory required is therefore the total number of parameters multiplied by the number of bytes this infomation occupies.

    - Adam stores two states per parameter, so the memory requirement for storing the information to compute the learning rate is 2 times the number of bytes required to store a parameter.

In [9]:
def calculate_optimizer_memory(param_count: float, bytes_per_param: int) -> float:
    """Calculates the memory in GB for Adam optimizer states.

    Args:
      param_count: The total number of parameters in the model.
      bytes_per_param: The number of bytes used to store a single parameter.

    Returns:
      The total memory required for optimizer states, in gigabytes.
    """
    total_bytes = 2 * param_count * bytes_per_param

    return formatting.bytes_to_gb(total_bytes)

In [10]:
feedback.test_calculate_optimizer_memory(calculate_optimizer_memory)

✅ Nice! Your answer looks correct.


**5. Calculate activation memory**

- When doing a forward pass, the model needs to store the results of each computation (also known as activations) in the GPU.


- The activation for a **single token** after passing through one transformer layer is a vector. The size of this vector is the embedding dimension.

    - The activation for a single token after passing through one transformer layer is a vector. The size of this vector is the embedding dimension.

    - memory for one token's activation=embedding dimension×bytes per parameter

- During the forward pass, this activation vector is stored for every single token in the sequence.

    - memory for one sequence=maximum sequence length×memory for one token's activation

- During training, the activations from every layer are kept in memory because they are needed for the backward pass.

    - memory for all layers=number of layers×memory for one sequence

- Finally, this entire process happens for every example in the batch simultaneously. The GPU must hold the activations for all of them at once.

    - total activation memory=batch Size×memory for all layers


In [11]:
def calculate_activation_memory(
    batch_size: int,
    max_length: int,
    num_layers: int,
    embedding_dim: int,
    bytes_per_param: int,
) -> float:
    """Estimates the memory in GB required for activations using a simplified
    formula.

    Args:
      batch_size: The number of sequences in the batch.
      max_length: The length of each sequence in tokens after padding.
      num_layers: The number of transformer layers in the model.
      embedding_dim: The hidden dimension size of the model.
      bytes_per_param: The number of bytes used for the activation values.

    Returns:
      The estimated total memory for activations, in gigabytes.
    """
    total_bytes = embedding_dim * bytes_per_param * max_length * num_layers * batch_size

    return formatting.bytes_to_gb(total_bytes)

In [12]:
feedback.test_calculate_activation_memory(calculate_activation_memory)

✅ Nice! Your answer looks correct.


**Computing total memory**

-

In [14]:
# Calculate memory for each component using your functions.
param_mem = calculate_param_memory(PARAM_COUNT, BYTES_PER_PARAMETER)

input_data_mem = calculate_input_data_memory(
    BATCH_SIZE, MAX_LENGTH, BYTES_PER_PARAMETER
)

grad_mem = calculate_gradient_memory(PARAM_COUNT, BYTES_PER_PARAMETER)

optim_mem = calculate_optimizer_memory(PARAM_COUNT, BYTES_PER_PARAMETER)

activ_mem = calculate_activation_memory(
    BATCH_SIZE, MAX_LENGTH, NUM_LAYERS, EMBEDDING_DIM, BYTES_PER_PARAMETER
)

total_inference_memory = param_mem + input_data_mem + activ_mem

# Sum them.
total_training_memory = total_inference_memory + grad_mem + optim_mem

# Display the results.
display(HTML("<h3>--- GPU memory consumption breakdown ---</h3>"))
display(HTML("<h4>------ During training and inference ------</h4>"))

# Use the existing function for each component.
formatting.display_memory("model parameters", param_mem)
formatting.display_memory("input data batch", input_data_mem, decimal_places=6)
formatting.display_memory("activations", activ_mem)

display(HTML("<h4>------ During training only ------</h4>"))
formatting.display_memory("gradients", grad_mem)
formatting.display_memory("optimizer states (Adam)", optim_mem)

# Display the formatted separator and total.
display(
    HTML(
        f"<h3>Total estimated GPU memory required during inference: "
        f"{total_inference_memory:.2f} GB</h3>"
    )
)
display(
    HTML(
        f"<h3>Total estimated GPU memory required during training or "
        f"fine-tuning: {total_training_memory:.2f} GB</h3>"
    )
)

**Solo: Reducing memory requirements with bfloat16**

- the parameters, gradients, and activations are stored as bfloat16 numbers instead of 32-bit floating point numbers.

In [15]:
# Set the number of bytes per parameter to 2 as bfloat16 uses only two bytes
# to store each parameter, gradient, and activation.
BYTES_PER_PARAMETER = 2

param_mem = calculate_param_memory(PARAM_COUNT, BYTES_PER_PARAMETER)
input_data_mem = calculate_input_data_memory(
    BATCH_SIZE, MAX_LENGTH, BYTES_PER_PARAMETER
)
grad_mem = calculate_gradient_memory(PARAM_COUNT, BYTES_PER_PARAMETER)
optim_mem = calculate_optimizer_memory(PARAM_COUNT, BYTES_PER_PARAMETER)
activ_mem = calculate_activation_memory(
    BATCH_SIZE, MAX_LENGTH, NUM_LAYERS, EMBEDDING_DIM, BYTES_PER_PARAMETER
)

total_inference_memory = param_mem + input_data_mem + activ_mem

# Sum them.
total_training_memory = total_inference_memory + grad_mem + optim_mem

# Display the results.
display(HTML("<h3>--- GPU memory consumption breakdown ---</h3>"))
display(HTML("<h4>------ During training and inference ------</h4>"))

# Use the existing function for each component.
formatting.display_memory("model parameters", param_mem)
formatting.display_memory("input data batch", input_data_mem, decimal_places=6)
formatting.display_memory("activations", activ_mem)
display(HTML("<h4>------ During training only ------</h4>"))
formatting.display_memory("gradients", grad_mem)
formatting.display_memory("optimizer states (Adam)", optim_mem)

# Display the formatted separator and total.
display(
    HTML(
        f"<h3>Total estimated GPU memory required during inference: "
        f"{total_inference_memory:.2f} GB</h3>"
    )
)
display(
    HTML(
        f"<h3>Total estimated GPU memory required during training or "
        f"fine-tuning: {total_training_memory:.2f} GB</h3>"
    )
)

The five main components that need to be stored in the GPU's memory are:

- Model parameters
- Input data
- Activations
- Gradients
- Optimizer states

